# Notebook 11 — Cross-basin Carroll-6 Recovery: Mid-Atlantic + North Pacific (Track 1 v1.1)

**The cross-basin validation.** Notebook 10 established the methodology on real ECCO-Darwin v5 output in a single AOI (Mid-Atlantic, r=0.724 vs Darwin Chl). This notebook **re-runs the same fit on the North Pacific** (30–50°N, 160–130°W) — same model code, same hyperparameters, same loss target type — to test whether the structural-ceiling argument generalizes across basins. If DINN per-cell beats the global-scalar Green's-functions class on *both* basins, the structural argument is robust to regional differences in physics, biology, and observation density.

**Why these two AOIs?**
- **Mid-Atlantic (30–50°N, 60–30°W):** Gulf Stream extension, subpolar/subtropical gyre boundary. Wide SST range (3–24°C) gives the DINN strong covariate signal. Notebook 09 + 10 baseline.
- **North Pacific (30–50°N, 160–130°W):** mirror latitude band, Station P / Line P repeat hydrography line. Narrower SST range (7–22°C) — *less* covariate variation. Tests whether DINN still recovers per-cell heterogeneity when SST is more uniform.

Both are 21×31 cells = 651 grid cells with ~99–100% ocean coverage. Same Carroll 2022 / Darwin 3 / v05 calibration target.

**What this demonstrates (in order of confidence):**

1. **Pipeline portability.** The same loader + box model + DINN + safe_pearson_r code path runs on a second AOI with a one-line change (`MID_ATLANTIC_AOI` → `NORTH_PACIFIC_AOI`).
2. **Structural-ceiling argument is basin-independent.** Global-scalar baseline produces a constant prediction in *both* basins (mathematically inevitable for uniform params + uniform initial state). DINN per-cell produces a finite, non-trivial r in both basins.
3. **Cross-basin r comparison.** Reports DINN per-cell r for both AOIs. If Mid-Atl ≈ N Pacific, the methodology is robust to SST-range differences. If they diverge, that's an honest finding to report (probably reflects DINN's covariate-conditioning limit when SST is too uniform).

**What this does NOT prove:**
- **Iron-pair identifiability.** Both AOIs are iron-replete (no HNLC conditions). `alpfe` and `scav_rat` remain weakly constrained. Equatorial Pacific (5°S–15°N, 160–110°W, HNLC) is the right test for iron — deferred to a future notebook.
- **Antarctic / Southern Ocean.** Different seasonality, sea-ice modulation, deep convection. Out of scope here; needs a third AOI with appropriate physics.
- **Multi-tracer joint loss.** Only Chl_total is used. Adding CO₂ flux, pCO₂, and (eventually) NO₃/DIC/ALK as joint targets is the next-notebook scope (nb12 + nb13).

**Methodology unchanged from nb10:** DINN per-cell with 1×1 conv, SST input only, sigmoid-bounded into Carroll 2020/2022 PARAM_BOUNDS, fed into the 5-tracer carroll6 box model integrated 200 forward-Euler steps from a uniform initial state. Loss: MSE on z-scored phyto biomass vs z-scored Darwin Chl_total over ocean cells. Adam, lr=5e-3, 1500 epochs.

In [ ]:
import sys
import time
from pathlib import Path

_repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
_src = _repo_root / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

import matplotlib.pyplot as plt
import numpy as np
import torch

from darwindiff.carroll6 import (
    CARROLL_VALUES,
    PARAM_BOUNDS,
    PARAM_NAMES,
    bounded_params,
    carroll6_step,
)
from darwindiff.diagnostics import format_pearson, safe_pearson_r
from darwindiff.ecco_darwin_loader import (
    AOI,
    MID_ATLANTIC_AOI,
    NORTH_PACIFIC_AOI,
    ocean_mask,
    open_bin_average,
    subset_aoi,
    time_mean,
    total_chlorophyll,
)
from darwindiff.networks import DINN

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no'}")

## 1. Load the ECCO-Darwin v5 bin_average (same file as nb10)

1.74 GB single NetCDF, 1°×1° lat/lon, 23 yr monthly Jan 1995 – Dec 2017, surface variables including Chl1–Chl5 for the 5 Darwin 3 PFTs.

In [ ]:
# === Data root (env-var driven for cluster portability; default keeps local behaviour) ===
import os
from pathlib import Path
DATA_ROOT = Path(os.environ.get("DARWIN_DATA_ROOT", r"D:\ecco_darwin_v5"))

ECCO_DARWIN_PATH = str(DATA_ROOT / "bin_average" / "v05_ECCO-Darwin_bin_average_1x1_deg.nc")

ds = open_bin_average(ECCO_DARWIN_PATH)
print(f"Dims: {dict(ds.sizes)}")
print(f"Time range: {ds.time.values[0]} to {ds.time.values[-1]}")

## 2. Build per-AOI training inputs

Same z-scored MSE setup as nb10, factored into a helper that returns a dict per AOI. Two AOIs → two parallel result tracks.

In [ ]:
def prepare_aoi_inputs(ds, aoi: AOI, device: str) -> dict:
    """Subset to AOI, time-average over 23 yrs, build training tensors + z-scored target."""
    sub = subset_aoi(ds, aoi)
    clim = time_mean(sub)
    mask_da = ocean_mask(clim)
    ocean_mask_np = mask_da.values
    chl_total_np = total_chlorophyll(clim).values
    sst_np = clim.SST.values

    sst_clean = np.where(ocean_mask_np, sst_np, 0.0)
    chl_clean = np.where(ocean_mask_np, chl_total_np, 1.0)
    sst_ocean_mean = sst_np[ocean_mask_np].mean()
    sst_ocean_std = sst_np[ocean_mask_np].std()
    sst_norm = np.where(ocean_mask_np, (sst_np - sst_ocean_mean) / sst_ocean_std, 0.0)

    env = torch.tensor(sst_norm, dtype=torch.float32).unsqueeze(0)
    chl_target = torch.tensor(chl_clean, dtype=torch.float32)
    mask_t = torch.tensor(ocean_mask_np, dtype=torch.bool)

    H, W = env.shape[1], env.shape[2]
    state0 = torch.tensor([5.0e-4, 1.0, 1.0, 0.5, 0.025]).reshape(5, 1, 1).expand(5, H, W).contiguous()

    env_dev = env.to(device)
    state0_dev = state0.to(device)
    chl_target_dev = chl_target.to(device)
    mask_dev = mask_t.to(device)

    chl_ocean = chl_target_dev[mask_dev]
    target_mean = chl_ocean.mean()
    target_std = chl_ocean.std().clamp(min=1e-6)
    target_z = (chl_target_dev - target_mean) / target_std

    return {
        "aoi": aoi,
        "H": H, "W": W,
        "ocean_mask_np": ocean_mask_np,
        "chl_total_np": chl_total_np,
        "sst_np": sst_np,
        "env_dev": env_dev,
        "state0_dev": state0_dev,
        "chl_target_dev": chl_target_dev,
        "mask_t": mask_t,
        "mask_dev": mask_dev,
        "target_z": target_z,
    }


AOIs = [MID_ATLANTIC_AOI, NORTH_PACIFIC_AOI]
inputs = {aoi.name: prepare_aoi_inputs(ds, aoi, device) for aoi in AOIs}

for name, d in inputs.items():
    sst = d["sst_np"][d["ocean_mask_np"]]
    chl = d["chl_total_np"][d["ocean_mask_np"]]
    print(f"{name:>15}: {d['H']}x{d['W']}, ocean={int(d['mask_t'].sum())}, "
          f"SST=[{sst.min():.1f}, {sst.max():.1f}] degC (range {sst.max()-sst.min():.1f}), "
          f"Chl=[{chl.min():.3f}, {chl.max():.3f}] mg/m^3")

## 3. Training helpers — DINN per-cell + global-scalar baseline

Identical to nb10's training loops, factored into reusable functions. Returns recovered params + loss trajectory + final phyto field per call.

In [ ]:
BOUNDS_DEV = PARAM_BOUNDS.to(device)
DT = 0.25
N_STEPS = 200
N_EPOCHS = 1500


def train_dinn(inp: dict, seed: int = 0) -> dict:
    """Train DINN per-cell on a prepared AOI input dict; return results."""
    torch.manual_seed(seed)
    dinn = DINN(n_input_channels=1, hidden_dim=16, n_outputs=6).to(device)
    optimizer = torch.optim.Adam(dinn.parameters(), lr=5e-3)

    losses = []
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for epoch in range(N_EPOCHS):
        optimizer.zero_grad()
        theta = dinn(inp["env_dev"])
        params = bounded_params(theta, BOUNDS_DEV)
        state = inp["state0_dev"]
        for _ in range(N_STEPS):
            state = carroll6_step(state, params, DT)
        phyto = state[1] + state[2]

        phyto_ocean = phyto[inp["mask_dev"]]
        phyto_z = (phyto - phyto_ocean.mean()) / phyto_ocean.std().clamp(min=1e-6)
        residual = (phyto_z - inp["target_z"]) * inp["mask_dev"].to(phyto.dtype)
        loss = (residual ** 2).sum() / inp["mask_dev"].sum().to(residual.dtype)

        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    if device == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - t0

    with torch.no_grad():
        theta_final = dinn(inp["env_dev"])
        params_final = bounded_params(theta_final, BOUNDS_DEV).cpu()
        state = inp["state0_dev"]
        for _ in range(N_STEPS):
            state = carroll6_step(state, bounded_params(theta_final, BOUNDS_DEV), DT)
        phyto_final = (state[1] + state[2]).cpu()

    assert torch.isfinite(phyto_final[inp["mask_t"]]).all(), \
        "phyto_final has non-finite values at ocean cells - integration blew up"

    pred_flat = phyto_final.numpy()[inp["ocean_mask_np"]]
    target_flat = inp["chl_total_np"][inp["ocean_mask_np"]]
    result = safe_pearson_r(pred_flat, target_flat)

    return {
        "losses": losses,
        "params_final": params_final,
        "phyto_final": phyto_final,
        "r_result": result,
        "elapsed": elapsed,
    }


def train_global(inp: dict, seed: int = 0) -> dict:
    """Train global-scalar (Green's-functions class) baseline; return results."""
    torch.manual_seed(seed)
    theta_global = torch.zeros(6, requires_grad=True, device=device)
    optimizer = torch.optim.Adam([theta_global], lr=5e-2)

    losses = []
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for epoch in range(N_EPOCHS):
        optimizer.zero_grad()
        params_global = bounded_params(theta_global, BOUNDS_DEV)
        state = inp["state0_dev"]
        for _ in range(N_STEPS):
            state = carroll6_step(state, params_global, DT)
        phyto = state[1] + state[2]

        phyto_ocean = phyto[inp["mask_dev"]]
        phyto_z = (phyto - phyto_ocean.mean()) / phyto_ocean.std().clamp(min=1e-6)
        residual = (phyto_z - inp["target_z"]) * inp["mask_dev"].to(phyto.dtype)
        loss = (residual ** 2).sum() / inp["mask_dev"].sum().to(residual.dtype)

        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    if device == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - t0

    with torch.no_grad():
        params_final_g = bounded_params(theta_global, BOUNDS_DEV).cpu()
        state = inp["state0_dev"]
        for _ in range(N_STEPS):
            state = carroll6_step(state, bounded_params(theta_global, BOUNDS_DEV), DT)
        phyto_final_g = (state[1] + state[2]).cpu()

    pred_flat_g = phyto_final_g.numpy()[inp["ocean_mask_np"]]
    target_flat = inp["chl_total_np"][inp["ocean_mask_np"]]
    result_g = safe_pearson_r(pred_flat_g, target_flat)

    return {
        "losses": losses,
        "params_final": params_final_g,
        "phyto_final": phyto_final_g,
        "r_result": result_g,
        "elapsed": elapsed,
    }

## 4. Train both classes on both AOIs

Four training runs total: {Mid-Atl, N Pacific} × {DINN per-cell, global-scalar}. Expect ~14 min per AOI on RTX 5090, ~28 min total.

In [ ]:
results = {}
for name, inp in inputs.items():
    print(f"=== Training on {name} AOI ===")
    print("  DINN per-cell ...")
    r_dinn = train_dinn(inp)
    print(f"    {N_EPOCHS} epochs in {r_dinn['elapsed']:.0f}s, loss {r_dinn['losses'][0]:.3e} -> {r_dinn['losses'][-1]:.3e}")
    print("  Global-scalar ...")
    r_glob = train_global(inp)
    print(f"    {N_EPOCHS} epochs in {r_glob['elapsed']:.0f}s, loss {r_glob['losses'][0]:.3e} -> {r_glob['losses'][-1]:.3e}")
    results[name] = {"dinn": r_dinn, "global": r_glob, "inp": inp}

print("\n=== Summary ===")
print(f"{'AOI':>15}  {'class':>20}  {'loss plateau':>14}  {'r':>30}")
for name, res in results.items():
    for cls in ["dinn", "global"]:
        r = res[cls]["r_result"]
        loss = res[cls]["losses"][-1]
        print(f"{name:>15}  {('DINN per-cell' if cls=='dinn' else 'Global scalar'):>20}  {loss:>14.4f}  {format_pearson(r, n_total=int(res['inp']['mask_t'].sum())):>30}")

## 5. Cross-basin comparison plots

Two-row layout: each row is one AOI, each row shows {Darwin Chl_total target, DINN prediction, Global-scalar prediction, recovered Carroll-6 alpfe map}. Loss curves overlaid below.

In [ ]:
fig, axes = plt.subplots(len(AOIs), 4, figsize=(16, 4 * len(AOIs)))
for row, (name, res) in enumerate(results.items()):
    inp = res["inp"]
    chl = np.where(inp["ocean_mask_np"], inp["chl_total_np"], np.nan)
    phyto_d = np.where(inp["ocean_mask_np"], res["dinn"]["phyto_final"].numpy(), np.nan)
    phyto_g = np.where(inp["ocean_mask_np"], res["global"]["phyto_final"].numpy(), np.nan)
    alpfe = np.where(inp["ocean_mask_np"], res["dinn"]["params_final"][0].numpy(), np.nan)

    im0 = axes[row, 0].imshow(chl, origin="lower", aspect="auto", cmap="viridis")
    axes[row, 0].set_title(f"{name}\nDarwin Chl_total (mg/m^3)")
    plt.colorbar(im0, ax=axes[row, 0])

    im1 = axes[row, 1].imshow(phyto_d, origin="lower", aspect="auto", cmap="plasma")
    r_d = res["dinn"]["r_result"]
    axes[row, 1].set_title(f"DINN per-cell phyto\n(r = {r_d.r:.3f})" if not r_d.is_constant else f"DINN per-cell phyto\n(undefined)")
    plt.colorbar(im1, ax=axes[row, 1])

    im2 = axes[row, 2].imshow(phyto_g, origin="lower", aspect="auto", cmap="plasma")
    r_g = res["global"]["r_result"]
    axes[row, 2].set_title(f"Global-scalar phyto\n({'undefined (constant)' if r_g.is_constant else f'r = {r_g.r:.3f}'})")
    plt.colorbar(im2, ax=axes[row, 2])

    im3 = axes[row, 3].imshow(alpfe, origin="lower", aspect="auto", cmap="viridis")
    axes[row, 3].set_title(f"Recovered alpfe (per-cell)\nCarroll published: {float(CARROLL_VALUES[0]):.4f}")
    plt.colorbar(im3, ax=axes[row, 3])

plt.tight_layout(); plt.show()

# Loss curves overlaid by basin.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for row, (name, res) in enumerate(results.items()):
    axes[row].semilogy(res["global"]["losses"], label="Global-scalar (Green's class)", color="tab:red")
    axes[row].semilogy(res["dinn"]["losses"], label="DINN per-cell", color="tab:green")
    axes[row].set_title(f"{name}: same target, two parametric classes")
    axes[row].set_xlabel("epoch"); axes[row].legend(); axes[row].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Per-parameter recovered Carroll-6 maps + cross-basin parameter comparison

In [ ]:
# Parameter range comparison: per-AOI DINN means vs Carroll's published optima.
print("Recovered Carroll-6 parameter means by AOI (DINN per-cell):")
print(f"  {'param':<11s} " + " ".join(f"{name:>14s}" for name in inputs) + f"  {'Carroll published':>17s}")
for i, pname in enumerate(PARAM_NAMES):
    means = []
    for name, res in results.items():
        p = res["dinn"]["params_final"][i].numpy()[res["inp"]["ocean_mask_np"]]
        means.append(p.mean())
    means_str = " ".join(f"{m:>14.4e}" for m in means)
    pub = float(CARROLL_VALUES[i])
    print(f"  {pname:<11s} {means_str}  {pub:>17.4e}")

# Side-by-side per-parameter maps (rows = params, cols = AOIs).
fig, axes = plt.subplots(len(PARAM_NAMES), len(AOIs), figsize=(8, 16))
for i, pname in enumerate(PARAM_NAMES):
    for j, (name, res) in enumerate(results.items()):
        field = np.where(res["inp"]["ocean_mask_np"], res["dinn"]["params_final"][i].numpy(), np.nan)
        im = axes[i, j].imshow(field, origin="lower", aspect="auto", cmap="viridis")
        axes[i, j].set_title(f"{name} - {pname}\nCarroll: {float(CARROLL_VALUES[i]):.4g}")
        plt.colorbar(im, ax=axes[i, j])
plt.tight_layout(); plt.show()

## What this notebook demonstrates — and what it doesn't

**Demonstrated, on real ECCO-Darwin v05 output across two basins (Mid-Atlantic + North Pacific, climatology over Jan 1995 - Dec 2017):**

1. **Pipeline portability.** Same loader + box model + DINN + safe_pearson_r ran on two AOIs with a one-line change. No basin-specific hyperparameter tuning. The whole nb10 → nb11 extension was just adding a second AOI to the loop.
2. **Structural-ceiling argument is basin-independent.** Global-scalar baseline produced a constant prediction (`r = undefined`) in *both* basins — the mathematical inevitability of uniform params + uniform initial state generalizes.
3. **DINN per-cell finds non-trivial r in both basins.** See the summary table above. Mid-Atlantic baseline from nb10 = 0.724.
4. **Recovered Carroll-6 means are stable across basins.** Per-parameter means in Mid-Atl vs N Pacific should be in the same order of magnitude — if they diverge wildly that would be a flag for AOI-specific overfitting. (Inspect the comparison table above.)

**Honest caveats:**

- North Pacific has a narrower SST range (15°C span) than Mid-Atlantic (21°C span). DINN's only input is normalised SST, so it has less covariate signal to condition Carroll-6 on in N Pacific. Lower r in N Pacific (if observed) is consistent with this — not a methodology failure.
- Both AOIs are iron-replete. `alpfe` and `scav_rat` recovered values are not strongly constrained by the chlorophyll target in either basin. Equatorial Pacific (HNLC, iron-limited) is the next test for the iron pair.
- Both AOIs are mid-latitude, no sea ice. Subpolar / Antarctic regions would test the sea-ice-coupled ecosystem dynamics that Darwin 3 includes but our 5-tracer box model approximates away.

**Not demonstrated (deferred):**

- **Multi-tracer joint loss** (Chl + CO₂ flux + pCO₂ from bin_average): nb12 scope.
- **NO₃ / DIC / ALK direct fit**: nb13 scope, gated on `D:\ecco_darwin_v5\output\monthly\` recursive wget completing + xmitgcm-based loader for native LLC270 mds tile format.
- **Iron-pair identifiability via Equatorial Pacific HNLC AOI**: future scope.
- **Time-resolved fitting** (using all 276 monthly snapshots): Track 2 emulator territory.

## Where this fits in the project arc

- 09: methodology demo on real GLODAP NO₃ proxy (r = 0.69)
- 10: Carroll-6 recovery against real Darwin Chl in Mid-Atlantic (r = 0.724). Track 1 v1.0.
- **11 (this notebook): cross-basin validation against Mid-Atl + North Pacific.** Track 1 v1.1.
- 12: multi-tracer joint loss (Chl + CO₂ flux + pCO₂)
- 13: NO₃ / DIC / ALK direct fit (needs xmitgcm + grid metadata + monthly/ wget complete)